# 02 — Correlation Analysis

**Project:** SERSA Product Demand Relationship Analysis  
**Author:** Cesar Enrique Banda Martinez  
**Date:** 2026  

---

## Context

This notebook is the second phase of the satellite project.  
It takes the monthly pivot table produced by `01_time_series_preparation.ipynb` and computes **Pearson correlation coefficients** across all SKUs to identify which products tend to be consumed together.

### What we are measuring
Pearson correlation between two SKUs measures whether their **monthly transaction counts move in the same direction** over time.  
- `r ≈ +1` → both SKUs rise and fall together (strong positive relationship).  
- `r ≈ -1` → when one rises, the other falls (strong negative relationship).  
- `r ≈ 0` → no linear relationship.

### Important limitation: spurious correlation from shared trend
If two SKUs simply grew over time alongside the overall business, their correlation will appear high even if there is no true demand relationship between them. This limitation is addressed in `03_growth_correlation.ipynb` using growth rates (`pct_change`).  
Here we compute the **raw contemporaneous correlation** as a first layer of the analysis.

### Goals of this notebook
1. Load `monthly_sales_pivot.csv`.
2. Filter out low-activity SKUs (< 10 total transactions across the full period).
3. Compute the Pearson correlation matrix.
4. Visualize the full correlation heatmap.
5. Extract and rank the strongest positive and negative pairs.
6. Export results for subsequent notebooks.

---

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

---
## 2. Configuration

In [ ]:
DATA_DIR       = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_TABLES  = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_FIGURES = os.path.join(os.getcwd(), "..", "outputs", "figures")

# SKUs with fewer than this many total transactions are excluded from correlation analysis
LOW_ACTIVITY_THRESHOLD = 10

# Correlation strength threshold for the top-pairs export
CORR_THRESHOLD = 0.75

print(f"Data dir:       {os.path.normpath(DATA_DIR)}")
print(f"Output tables:  {os.path.normpath(OUTPUT_TABLES)}")
print(f"Output figures: {os.path.normpath(OUTPUT_FIGURES)}")

---
## 3. Load Monthly Pivot

In [ ]:
pivot = pd.read_csv(
    os.path.join(DATA_DIR, "monthly_sales_pivot.csv"),
    index_col="Month",
    parse_dates=True,
    decimal=","
)

print(f"Loaded pivot: {pivot.shape[0]} months x {pivot.shape[1]} SKUs")
print()
print(pivot.iloc[:3, :6])

---
## 4. Filter Low-Activity SKUs

Six SKUs were identified in notebook 01 with fewer than 10 total transactions across the entire 4-year period.  
These products lack sufficient signal for meaningful correlation analysis — a SKU dispensed 1–4 times in 50 months produces a near-zero vector that inflates or deflates correlations artificially.  
They are excluded here and documented below.

In [ ]:
sku_totals = pivot.sum()
low_activity = sku_totals[sku_totals < LOW_ACTIVITY_THRESHOLD].sort_values()

print(f"SKUs excluded (< {LOW_ACTIVITY_THRESHOLD} total transactions):")
print(low_activity.to_string())
print()

pivot_filtered = pivot[sku_totals[sku_totals >= LOW_ACTIVITY_THRESHOLD].index]

print(f"Pivot after filtering: {pivot_filtered.shape[0]} months x {pivot_filtered.shape[1]} SKUs")

---
## 5. Pearson Correlation Matrix

We compute pairwise Pearson correlation between all remaining SKUs using their monthly transaction counts.

In [ ]:
corr_matrix = pivot_filtered.corr(method="pearson")

print(f"Correlation matrix shape: {corr_matrix.shape}")
print()
print(corr_matrix.iloc[:4, :4].round(3))

---
## 6. Full Correlation Heatmap

The heatmap shows pairwise correlations across all 86 SKUs.  
At this scale, the goal is to identify **clusters** (groups of highly correlated SKUs) and **anti-correlated** regions, rather than reading individual values.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 16))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="RdYlGn",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0,
    square=True,
    cbar_kws={"shrink": 0.6, "label": "Pearson r"},
    ax=ax
)

ax.set_title("SKU Correlation Matrix — Monthly Transaction Counts", fontsize=16, fontweight="bold", pad=16)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelsize=6, rotation=90)
ax.tick_params(axis="y", labelsize=6, rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "02_correlation_heatmap_full.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 02_correlation_heatmap_full.png")

---
## 7. Top Correlation Pairs

We extract all unique SKU pairs with correlation above and below the defined thresholds,  
ranking them to identify the strongest demand relationships in the catalog.

In [ ]:
# Extract upper triangle only (avoid duplicate pairs and self-correlations)
corr_unstacked = (
    corr_matrix
    .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_unstacked.columns = ["SKU_A", "SKU_B", "Pearson_r"]
corr_unstacked["Pearson_r"] = corr_unstacked["Pearson_r"].round(4)

# Top positive pairs
top_positive = (
    corr_unstacked[corr_unstacked["Pearson_r"] >= CORR_THRESHOLD]
    .sort_values("Pearson_r", ascending=False)
    .reset_index(drop=True)
)

# Top negative pairs
top_negative = (
    corr_unstacked[corr_unstacked["Pearson_r"] <= -CORR_THRESHOLD]
    .sort_values("Pearson_r", ascending=True)
    .reset_index(drop=True)
)

print(f"Pairs with r >= {CORR_THRESHOLD}  : {len(top_positive)}")
print(f"Pairs with r <= -{CORR_THRESHOLD} : {len(top_negative)}")
print()
print("Top 15 positive pairs:")
print(top_positive.head(15).to_string(index=False))

In [ ]:
print("Top negative pairs:")
if len(top_negative) > 0:
    print(top_negative.to_string(index=False))
else:
    print(f"No pairs with r <= -{CORR_THRESHOLD} found.")

---
## 8. Top Pairs Bar Chart

Visualizing the 20 strongest positive correlations makes it easier to communicate findings.

In [ ]:
top_n = min(20, len(top_positive))

if top_n > 0:
    plot_data = top_positive.head(top_n).copy()
    plot_data["pair"] = plot_data["SKU_A"] + "  \u2194  " + plot_data["SKU_B"]

    fig, ax = plt.subplots(figsize=(10, top_n * 0.45 + 1))

    bars = ax.barh(
        plot_data["pair"][::-1],
        plot_data["Pearson_r"][::-1],
        color="#2C7BB6",
        alpha=0.85
    )

    ax.axvline(x=CORR_THRESHOLD, color="#D7191C", linestyle="--", linewidth=1, label=f"Threshold r={CORR_THRESHOLD}")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Pearson r", fontsize=11)
    ax.set_title(f"Top {top_n} Strongest Positive SKU Correlations", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.3)
    sns.despine()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FIGURES, "02_top_positive_pairs.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: 02_top_positive_pairs.png")
else:
    print(f"No pairs above threshold {CORR_THRESHOLD} to plot.")

---
## 9. Export

In [ ]:
corr_matrix.to_csv(
    os.path.join(OUTPUT_TABLES, "02_correlation_matrix.csv"),
    decimal=","
)

top_positive.to_csv(
    os.path.join(OUTPUT_TABLES, "02_top_positive_pairs.csv"),
    index=False,
    decimal=","
)

top_negative.to_csv(
    os.path.join(OUTPUT_TABLES, "02_top_negative_pairs.csv"),
    index=False,
    decimal=","
)

pivot_filtered.to_csv(
    os.path.join(OUTPUT_TABLES, "02_pivot_filtered.csv"),
    decimal=","
)

print("Export complete.")
print(f"  02_correlation_matrix.csv    -> {corr_matrix.shape}")
print(f"  02_top_positive_pairs.csv    -> {len(top_positive)} pairs")
print(f"  02_top_negative_pairs.csv    -> {len(top_negative)} pairs")
print(f"  02_pivot_filtered.csv        -> {pivot_filtered.shape[0]} months x {pivot_filtered.shape[1]} SKUs")

---
## 10. Summary

| Output | Description |
|--------|-------------|
| `02_correlation_matrix.csv` | Full 86 × 86 Pearson correlation matrix |
| `02_top_positive_pairs.csv` | All pairs with r ≥ 0.75 |
| `02_top_negative_pairs.csv` | All pairs with r ≤ −0.75 |
| `02_pivot_filtered.csv` | Pivot with low-activity SKUs removed (used by notebooks 03–05) |
| `02_correlation_heatmap_full.png` | Full heatmap — 86 SKUs |
| `02_top_positive_pairs.png` | Top 20 positive pairs bar chart |

### Limitation carried forward
High raw correlations may reflect **shared growth trend** rather than true demand co-movement.  
`03_growth_correlation.ipynb` addresses this by computing correlations on **monthly growth rates** (`pct_change`), which removes the common upward trend and isolates genuine co-movement signals.

**Next:** `03_growth_correlation.ipynb` — correlation on detrended growth rates.